In [13]:
import pandas as pd
import numpy as np
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

In [14]:
df = pd.read_csv('/content/IMDB Dataset.csv')

In [15]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [16]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [17]:
text = df.loc[0, df.columns[0]]
tokens = word_tokenize(text)
filtered = [word.lower() for word in tokens if word.lower() not in stop_words]
lemmatized = [lemmatizer.lemmatize(word) for word in filtered]
print(' '.join(lemmatized))

one reviewer mentioned watching 1 oz episode 'll hooked . right , exactly happened me. < br / > < br / > first thing struck oz brutality unflinching scene violence , set right word go . trust , show faint hearted timid . show pull punch regard drug , sex violence . hardcore , classic use word. < br / > < br / > called oz nickname given oswald maximum security state penitentary . focus mainly emerald city , experimental section prison cell glass front face inwards , privacy high agenda . em city home many .. aryan , muslim , gangsta , latino , christian , italian , irish .... scuffle , death stare , dodgy dealing shady agreement never far away. < br / > < br / > would say main appeal show due fact go show would n't dare . forget pretty picture painted mainstream audience , forget charm , forget romance ... oz n't mess around . first episode ever saw struck nasty surreal , could n't say ready , watched , developed taste oz , got accustomed high level graphic violence . violence , injusti

In [18]:
df['review'] = df['review'].str.replace('<br />', '', regex=False)
df['review'] = df['review'].str.replace('<br/>', '', regex=False)

In [19]:
for i in range(len(df)):
    text = df.loc[i, df.columns[0]]
    tokens = word_tokenize(text)
    # Добавлено условие word.isalpha() — убирает знаки препинания
    filtered = [word.lower() for word in tokens if word.lower() not in stop_words and word.isalpha()]
    lemmatized = [lemmatizer.lemmatize(word) for word in filtered]
    df.loc[i, 'processed_review'] = ' '.join(lemmatized)

print(df.head())

                                              review sentiment  \
0  One of the other reviewers has mentioned that ...  positive   
1  A wonderful little production. The filming tec...  positive   
2  I thought this was a wonderful way to spend ti...  positive   
3  Basically there's a family where a little boy ...  negative   
4  Petter Mattei's "Love in the Time of Money" is...  positive   

                                    processed_review  
0  one reviewer mentioned watching oz episode hoo...  
1  wonderful little production filming technique ...  
2  thought wonderful way spend time hot summer we...  
3  basically family little boy jake think zombie ...  
4  petter mattei love time money visually stunnin...  


In [20]:
df['label'] = df['sentiment'].map({'positive': 1, 'negative': 0})

In [21]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import numpy as np

In [22]:
X = df['processed_review']
y = df['label']

In [23]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [24]:
print("=== Bag of Words ===")
count_vec = CountVectorizer(max_df=0.95, min_df=2, max_features=5000)
X_train_count = count_vec.fit_transform(X_train)
X_test_count = count_vec.transform(X_test)

model = LogisticRegression()
model.fit(X_train_count, y_train)
y_pred = model.predict(X_test_count)
print(f"Точность (accuracy): {accuracy_score(y_test, y_pred):.4f}\n")

=== Bag of Words ===
Точность (accuracy): 0.8696



/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [25]:
print("=== TF-IDF ===")
tfidf_vec = TfidfVectorizer(max_df=0.95, min_df=2, max_features=5000)
X_train_tfidf = tfidf_vec.fit_transform(X_train)
X_test_tfidf = tfidf_vec.transform(X_test)

model = LogisticRegression()
model.fit(X_train_tfidf, y_train)
y_pred = model.predict(X_test_tfidf)
print(f"Точность (accuracy): {accuracy_score(y_test, y_pred):.4f}\n")

=== TF-IDF ===
Точность (accuracy): 0.8857



In [26]:
pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 94.8 MB/s eta 0:00:00


In [27]:
print("=== Word2Vec ===")
from gensim.models import Word2Vec

sentences = [text.split() for text in X_train]

w2v_model = Word2Vec(sentences, vector_size=100, window=5, min_count=2, workers=4)

# Функция для получения среднего вектора всего отзыва
def get_document_vector(text):
    words = text.split()
    word_vectors = [w2v_model.wv[word] for word in words if word in w2v_model.wv]
    if len(word_vectors) == 0:
        return np.zeros(w2v_model.vector_size)
    return np.mean(word_vectors, axis=0)

# Преобразуем все тексты в векторы
X_train_w2v = np.array([get_document_vector(text) for text in X_train])
X_test_w2v = np.array([get_document_vector(text) for text in X_test])

model = LogisticRegression()
model.fit(X_train_w2v, y_train)
y_pred = model.predict(X_test_w2v)
print(f"Точность (accuracy): {accuracy_score(y_test, y_pred):.4f}")

=== Word2Vec ===
Точность (accuracy): 0.8578


In [28]:
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb

In [29]:
print("\n=== Random Forest (TF-IDF) ===")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train_tfidf, y_train)
y_pred_rf = rf_model.predict(X_test_tfidf)
print(f"Точность (accuracy): {accuracy_score(y_test, y_pred_rf):.4f}")

print("\n=== XGBoost (TF-IDF) ===")
xgb_model = xgb.XGBClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
xgb_model.fit(X_train_tfidf, y_train)
y_pred_xgb = xgb_model.predict(X_test_tfidf)
print(f"Точность (accuracy): {accuracy_score(y_test, y_pred_xgb):.4f}")


=== Random Forest (TF-IDF) ===
Точность (accuracy): 0.8497

=== XGBoost (TF-IDF) ===
Точность (accuracy): 0.8308


In [30]:
X_train_tokens = [text.split() for text in X_train]
X_test_tokens = [text.split() for text in X_test]

In [31]:
print("=== N-Grams ===")
print("Учитывает порядок слов: биграммы, триграммы (улучшает Bag of Words).")

from sklearn.feature_extraction.text import CountVectorizer
ngram_vec = CountVectorizer(ngram_range=(1, 3), max_df=0.95, min_df=2, max_features=5000)
X_train_ngram = ngram_vec.fit_transform(X_train)
X_test_ngram = ngram_vec.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_ngram, y_train)
y_pred = model.predict(X_test_ngram)
print(f"Точность (accuracy): {accuracy_score(y_test, y_pred):.4f}\n")

=== N-Grams ===
Учитывает порядок слов: биграммы, триграммы (улучшает Bag of Words).
Точность (accuracy): 0.8744



In [32]:
print("=== HashingVectorizer ===")
print("Хэширует слова, не хранит словарь. Экономит память, но векторы не интерпретируемы.")

from sklearn.feature_extraction.text import HashingVectorizer
hash_vec = HashingVectorizer(n_features=2**18, alternate_sign=False)
X_train_hash = hash_vec.transform(X_train)
X_test_hash = hash_vec.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_hash, y_train)
y_pred = model.predict(X_test_hash)
print(f"Точность (accuracy): {accuracy_score(y_test, y_pred):.4f}\n")

=== HashingVectorizer ===
Хэширует слова, не хранит словарь. Экономит память, но векторы не интерпретируемы.
Точность (accuracy): 0.8833



In [36]:
print("=== GloVe ===")
print("Глобальная статистика встречаемости слов. Используем предобученную модель (wiki-gigaword).")

import gensim.downloader as api
import numpy as np

glove = api.load("glove-wiki-gigaword-50")  # 50-мерные векторы

def doc_glove_vector(text):
    words = text.split()
    vecs = [glove[word] for word in words if word in glove]
    return np.mean(vecs, axis=0) if vecs else np.zeros(50)

X_train_glove = np.array([doc_glove_vector(text) for text in X_train])
X_test_glove = np.array([doc_glove_vector(text) for text in X_test])

model = LogisticRegression(max_iter=1000)
model.fit(X_train_glove, y_train)
y_pred = model.predict(X_test_glove)
print(f"Точность (accuracy): {accuracy_score(y_test, y_pred):.4f}\n")

=== GloVe ===
Глобальная статистика встречаемости слов. Используем предобученную модель (wiki-gigaword).
[==================================================] 100.0% 66.0/66.0MB downloaded
Точность (accuracy): 0.7542



In [38]:
pip install fasttext

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 kB 3.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached pybind11-3.0.3-py3-none-any.whl.metadata (10 kB)
Using cached pybind11-3.0.3-py3-none-any.whl (313 kB)
  Created wheel for fasttext: filename=fasttext-0.9.3-cp312-cp312-linux_x86_64.whl size=4653976 sha256=cc525c7f6fda9b337a5537d1ff652d259e4eb4a1a129c5fdfe94eaf486d04a79
  Stored in directory: /root/.cache/pip/wheels/20/27/95/a7baf1b435f1cbde017cabdf1e9688526d2b0e929255a359c6
Successfully built fasttext


In [39]:
print("=== FastText ===")
print("Работает с редкими словами и морфологией через символьные n-граммы.")

import fasttext
import tempfile
import numpy as np

with tempfile.NamedTemporaryFile(mode='w', delete=False) as f:
    for text in X_train:
        f.write(text + '\n')
    temp_path = f.name

ft_model = fasttext.train_unsupervised(temp_path, model='cbow', dim=100)

def doc_fasttext_vector(text):
    words = text.split()
    vecs = [ft_model.get_word_vector(w) for w in words]
    return np.mean(vecs, axis=0) if vecs else np.zeros(100)

X_train_ft = np.array([doc_fasttext_vector(text) for text in X_train])
X_test_ft = np.array([doc_fasttext_vector(text) for text in X_test])

model = LogisticRegression(max_iter=1000)
model.fit(X_train_ft, y_train)
y_pred = model.predict(X_test_ft)
print(f"Точность (accuracy): {accuracy_score(y_test, y_pred):.4f}\n")

=== FastText ===
Работает с редкими словами и морфологией через символьные n-граммы.
Точность (accuracy): 0.8497



In [40]:
print("=== Doc2Vec ===")
print("Обучает вектор для целого документа, учитывая порядок слов.")

from gensim.models.doc2vec import Doc2Vec, TaggedDocument

train_tagged = [TaggedDocument(words=doc, tags=[str(i)]) for i, doc in enumerate(X_train_tokens)]
test_tagged = [TaggedDocument(words=doc, tags=[str(i)]) for i, doc in enumerate(X_test_tokens)]

d2v = Doc2Vec(vector_size=100, window=5, min_count=2, epochs=40)
d2v.build_vocab(train_tagged)
d2v.train(train_tagged, total_examples=d2v.corpus_count, epochs=d2v.epochs)

X_train_d2v = np.array([d2v.infer_vector(doc) for doc in X_train_tokens])
X_test_d2v = np.array([d2v.infer_vector(doc) for doc in X_test_tokens])

model = LogisticRegression(max_iter=1000)
model.fit(X_train_d2v, y_train)
y_pred = model.predict(X_test_d2v)
print(f"Точность (accuracy): {accuracy_score(y_test, y_pred):.4f}\n")

=== Doc2Vec ===
Обучает вектор для целого документа, учитывая порядок слов.
Точность (accuracy): 0.8581



In [48]:
print("=== Sentence Transformers (SBERT) ===")
print("Использует BERT для получения смыслового вектора предложения.")

# pip install sentence-transformers
from sentence_transformers import SentenceTransformer

# Загружаем лёгкую мультиязычную модель (для английского можно 'all-MiniLM-L6-v2')
model_sbert = SentenceTransformer('all-MiniLM-L6-v2')

X_train_sbert = model_sbert.encode(X_train.tolist(), show_progress_bar=True)
X_test_sbert = model_sbert.encode(X_test.tolist(), show_progress_bar=True)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_sbert, y_train)
y_pred = model.predict(X_test_sbert)
print(f"Точность (accuracy): {accuracy_score(y_test, y_pred):.4f}\n")

=== Sentence Transformers (SBERT) ===
Использует BERT для получения смыслового вектора предложения.


model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1250 [00:00<?, ?it/s]

Batches:   0%|          | 0/313 [00:00<?, ?it/s]

Точность (accuracy): 0.8165



In [42]:
print("=== One-Hot Encoding ===")
print("Каждое слово кодируется единичным вектором. Простой, но очень размерный.")

from sklearn.preprocessing import OneHotEncoder
import numpy as np

all_words = set()
for tokens in X_train_tokens:
    all_words.update(tokens)
word_to_idx = {word: i for i, word in enumerate(all_words)}
vocab_size = len(all_words)

def one_hot_doc(tokens):
    vec = np.zeros(vocab_size)
    for t in tokens:
        if t in word_to_idx:
            vec[word_to_idx[t]] = 1  # бинарный (можно считать частоту)
    return vec

X_train_ohe = np.array([one_hot_doc(doc) for doc in X_train_tokens])
X_test_ohe = np.array([one_hot_doc(doc) for doc in X_test_tokens])

model = LogisticRegression(max_iter=1000)
model.fit(X_train_ohe, y_train)
y_pred = model.predict(X_test_ohe)
print(f"Точность (accuracy): {accuracy_score(y_test, y_pred):.4f}\n")

=== One-Hot Encoding ===
Каждое слово кодируется единичным вектором. Простой, но очень размерный.
Точность (accuracy): 0.8792



## spacy

In [43]:
pip install spacy


In [44]:
import spacy


In [45]:
df2 = pd.read_csv('/content/IMDB Dataset.csv')

In [46]:
# Загружаем модель spaCy для английского
nlp = spacy.load("en_core_web_sm")

# Удаляем теги <br /> и <br/> из исходного текста (в df2)
df2['review'] = df2['review'].str.replace('<br />', '', regex=False)
df2['review'] = df2['review'].str.replace('<br/>', '', regex=False)

# Создаём пустую колонку для обработанных текстов
df2['processed_review'] = ""

# Обрабатываем каждую строку через цикл
for i in range(len(df2)):
    text = df2.loc[i, df2.columns[0]]  # первый столбец (review)

    # Обрабатываем текст через пайплайн spaCy
    doc = nlp(text)

    # Собираем леммы: только слова, не стоп-слова, только буквы, нижний регистр
    lemmas = [token.lemma_.lower() for token in doc
              if not token.is_stop and token.is_alpha]

    # Сохраняем обработанный текст
    df2.loc[i, 'processed_review'] = ' '.join(lemmas)

# Проверяем результат
print(df2.head())

# Преобразуем метки: positive → 1, negative → 0
df2['label'] = df2['sentiment'].map({'positive': 1, 'negative': 0})

                                              review sentiment  \
0  One of the other reviewers has mentioned that ...  positive   
1  A wonderful little production. The filming tec...  positive   
2  I thought this was a wonderful way to spend ti...  positive   
3  Basically there's a family where a little boy ...  negative   
4  Petter Mattei's "Love in the Time of Money" is...  positive   

                                    processed_review  
0  reviewer mention watch oz episode hook right e...  
1  wonderful little production filming technique ...  
2  think wonderful way spend time hot summer week...  
3  basically family little boy jake think zombie ...  
4  petter mattei love time money visually stunnin...  


In [47]:
df2

,review,sentiment,processed_review,label
0,One of the other reviewers has mentioned that ...,positive,reviewer mention watch oz episode hook right e...,1
1,A wonderful little production. The filming tec...,positive,wonderful little production filming technique ...,1
2,I thought this was a wonderful way to spend ti...,positive,think wonderful way spend time hot summer week...,1
3,Basically there's a family where a little boy ...,negative,basically family little boy jake think zombie ...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,petter mattei love time money visually stunnin...,1
...,...,...,...,...
49995,I thought this movie did a down right good job...,positive,think movie right good job creative original e...,1
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative,bad plot bad dialogue bad acting idiotic direc...,0
49997,I am a Catholic taught in parochial elementary...,negative,catholic teach parochial elementary school nun...,0
49998,I'm going to have to disagree with the previou...,negative,go disagree previous comment maltin second rat...,0
